In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Install the SQL Server Python driver

In [2]:
%pip install pyodbc sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
from sqlalchemy import create_engine
import urllib

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-R1TVLSN\SQLEXPRESS;"
    "DATABASE=ProfitLeakAnalysis;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

## Test the connection

In [4]:
test = pd.read_sql(
    "SELECT TOP 5 * FROM analytics.OrderProfitabilityDetail",
    engine
)

test

D:\ana 2\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


,order_id,customer_id,order_date,sales_channel,device_type,shipping_method,coupon_code,discount_rate,product_id,product_name,category,subcategory,brand,order_item_id,quantity,unit_list_price,unit_selling_price,discount_amount,unit_cost,gross_revenue,product_cost,ProductMargin
0,O0000017,C000626,2024-02-24,Website,Mobile,Standard,WELCOME10,0.05,P00470,Nova Chargers 0470,Electronics,Chargers,Nova,OI00000024,1,64.35,60.52,3.83,16.01,60.52,16.01,44.51
1,O0000017,C000626,2024-02-24,Website,Mobile,Standard,WELCOME10,0.05,P00252,Ridge Jackets 0252,Apparel,Jackets,Ridge,OI00000025,2,18.78,17.99,0.79,9.54,35.98,19.08,16.90
2,O0000018,C014363,2024-06-03,Website,Mobile,Express,SPRING15,0.15,P00325,Vertex Dresses 0325,Apparel,Dresses,Vertex,OI00000026,2,31.03,26.21,4.82,19.79,52.42,39.58,12.84
3,O0000019,C012396,2024-05-04,Marketplace,Desktop,Standard,SPRING15,0.15,P00545,Everline Jeans 0545,Apparel,Jeans,Everline,OI00000027,1,61.86,52.75,9.11,27.45,52.75,27.45,25.30
4,O0000020,C027727,2024-05-27,Website,Mobile,Next-Day,WELCOME10,0.05,P00653,Luma Activewear 0653,Apparel,Activewear,Luma,OI00000028,2,10.19,9.69,0.50,4.88,19.38,9.76,9.62


## Load all four analytical datasets (views)

In [5]:
#View 1:OrderProfitabilityDetail
order_profitability = pd.read_sql(
    """
    SELECT *
    FROM analytics.OrderProfitabilityDetail
    """,
    engine
)

#View 2: Customer Economics
customer_economics = pd.read_sql(
    """
    SELECT *
    FROM analytics.CustomerEconomics
    """,
    engine
)

##View 3:Fulfillment Analysis
fulfillment = pd.read_sql(
    """
    SELECT *
    FROM analytics.FulfillmentAnalysis
    """,
    engine
)

# View 4:Marketing Economics
marketing = pd.read_sql(
    """
    SELECT *
    FROM analytics.MarketingEconomics
    """,
    engine
)

#View 5:Validated Returns
returns = pd.read_sql(
    """
    SELECT *
    FROM analytics.ValidatedReturns
    """,
    engine
)

# Check the dataset shapes and info

In [6]:
print("Order profitability:", order_profitability.shape)
print("Customer economics:", customer_economics.shape)
print("Fulfillment:", fulfillment.shape)
print("Marketing:", marketing.shape)
print("Returns:", returns.shape)

Order profitability: (101887, 22)
Customer economics: (30000, 22)
Fulfillment: (30000, 18)
Marketing: (5644, 17)
Returns: (14479, 17)


In [7]:
order_profitability.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101887 entries, 0 to 101886
Data columns (total 22 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   order_id            101887 non-null  object 
 1   customer_id         101887 non-null  object 
 2   order_date          101887 non-null  object 
 3   sales_channel       101887 non-null  object 
 4   device_type         101887 non-null  object 
 5   shipping_method     101887 non-null  object 
 6   coupon_code         101887 non-null  object 
 7   discount_rate       101887 non-null  float64
 8   product_id          101887 non-null  object 
 9   product_name        101887 non-null  object 
 10  category            101887 non-null  object 
 11  subcategory         101887 non-null  object 
 12  brand               101887 non-null  object 
 13  order_item_id       101887 non-null  object 
 14  quantity            101887 non-null  int64  
 15  unit_list_price     101887 non-nul

In [8]:
customer_economics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 22 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   customer_id                         30000 non-null  object 
 1   signup_date                         30000 non-null  object 
 2   acquisition_channel                 30000 non-null  object 
 3   customer_type                       30000 non-null  object 
 4   country                             30000 non-null  object 
 5   state                               30000 non-null  object 
 6   city                                30000 non-null  object 
 7   age_band                            30000 non-null  object 
 8   income_band                         30000 non-null  object 
 9   loyalty_tier                        30000 non-null  object 
 10  CompletedOrders                     30000 non-null  int64  
 11  Revenue                             30000

## Convert dates properly

In [9]:
date_columns = {
    "order_profitability": ["order_date"],
    "customer_economics": [
        "signup_date",
        "FirstOrderDate",
        "LastOrderDate"
    ],
    "fulfillment": [
        "order_date",
        "ship_date",
        "promised_date",
        "delivery_date"
    ],
    "marketing": ["date"],
    "returns": ["return_date"]
}

for column in date_columns["order_profitability"]:
    order_profitability[column] = pd.to_datetime(
        order_profitability[column]
    )

for column in date_columns["customer_economics"]:
    customer_economics[column] = pd.to_datetime(
        customer_economics[column]
    )

for column in date_columns["fulfillment"]:
    fulfillment[column] = pd.to_datetime(
        fulfillment[column]
    )

marketing["date"] = pd.to_datetime(marketing["date"])

returns["return_date"] = pd.to_datetime(
    returns["return_date"]
)

## Checking duplicates

In [10]:
print(
    "Duplicate order items:",
    order_profitability["order_item_id"].duplicated().sum()
)

print(
    "Duplicate customers:",
    customer_economics["customer_id"].duplicated().sum()
)

print(
    "Duplicate shipments:",
    fulfillment["shipment_id"].duplicated().sum()
)

print(
    "Duplicate marketing campaign-days:",
    marketing.duplicated(
        subset=["date", "campaign_id"]
    ).sum()
)

Duplicate order items: 0
Duplicate customers: 0
Duplicate shipments: 0
Duplicate marketing campaign-days: 0


## Check some important business metrices

In [11]:
print(
    "Revenue:",
    order_profitability["gross_revenue"].sum()
)

print(
    "Product cost:",
    order_profitability["product_cost"].sum()
)

print(
    "Product margin:",
    order_profitability["ProductMargin"].sum()
)

Revenue: 7998583.140000001
Product cost: 3931229.6899999995
Product margin: 4067353.4499999997


# DEEP ANALYSIS

### Build profitability baseline

#### Research Question:
        How much value remains after the major direct costs represented in the dataset?

In [12]:
baseline={
    "Revenue":order_profitability['gross_revenue'].sum(),
    "ProductCost":order_profitability['product_cost'].sum(),
    "ProductMargin":order_profitability['ProductMargin'].sum(),
    "Refunds":returns['valid_refund_amount'].sum(),
    "ReturnShipping":returns['valid_return_shipping_cost'].sum(),
    "Restocking":returns['valid_restocking_cost'].sum(),
    "OutboundShipping":fulfillment['shipping_cost'].sum()
}

baseline

{'Revenue': np.float64(7998583.140000001),
 'ProductCost': np.float64(3931229.6899999995),
 'ProductMargin': np.float64(4067353.4499999997),
 'Refunds': np.float64(641480.6799999999),
 'ReturnShipping': np.float64(115163.74),
 'Restocking': np.float64(25674.730000000003),
 'OutboundShipping': np.float64(283476.86)}

In [13]:
#Calculate contribution

contribution_profit=(
    baseline['Revenue']
    -baseline['ProductCost']
    -baseline['Refunds']
    -baseline['ReturnShipping']
    -baseline['Restocking']
    -baseline['OutboundShipping']
)

contribution_margin=(contribution_profit / baseline['Revenue'] * 100 )

print(f"Contribution profit: ${contribution_profit:,.2f}")
print(f"Contribution margin: {contribution_margin:.2f}%")

Contribution profit: $3,001,557.44
Contribution margin: 37.53%


## Revenue and profitability trend

### Research question
    Did revenue and product economics improve or deteriorate over time?

In [14]:
order_profitability["order_date"] = pd.to_datetime(
    order_profitability["order_date"]
)

order_profitability["year"] = (
    order_profitability["order_date"].dt.year
)

In [15]:
yearly=(
    order_profitability
    .groupby('year')
    .agg(
        Revenue=('gross_revenue','sum'),
        ProductCost=('product_cost','sum'),
        ProductMargin=('ProductMargin','sum'),
        Orders=('order_id','nunique')
    ).reset_index()
)

#MarginPct
yearly["MarginPct"] = (
    yearly["ProductMargin"]
    / yearly["Revenue"]
    * 100
)

#RevenueGrowth
yearly["RevenueGrowthPct"] = (
    yearly["Revenue"].pct_change() * 100
)

yearly

,year,Revenue,ProductCost,ProductMargin,Orders,MarginPct,RevenueGrowthPct
0,2024,"4,035,042.19","1,984,582.97","2,050,459.22",36591,50.82,NaN
1,2025,"3,963,540.95","1,946,646.72","2,016,894.23",36181,50.89,-1.77


### Conclusion:
    Revenue declined slightly in 2025, while product margin percentage remained almost unchanged. The bigger deterioration appears after product margin, where return and fulfillment costs need to be considered.

## Contribution Profitability by Year
    Did contribution profitability improve or deteriorate over time?

In [16]:
# ============================================================
# CONTRIBUTION PROFITABILITY BY ORDER YEAR
# ============================================================

# Sales by order year
sales_year = (
    order_profitability
    .assign(year=pd.to_datetime(order_profitability["order_date"]).dt.year)
    .groupby("year")
    .agg(
        Revenue=("gross_revenue", "sum"),
        ProductCost=("product_cost", "sum")
    )
    .reset_index()
)

sales_year["ProductMargin"] = (
    sales_year["Revenue"] -
    sales_year["ProductCost"]
)


# Returns assigned to the year of the original order
returns_year = (
    returns
    .merge(
        order_profitability[
            ["order_id", "order_date"]
        ].drop_duplicates("order_id"),
        on="order_id",
        how="inner",
        validate="many_to_one"
    )
)

returns_year["year"] = (
    pd.to_datetime(returns_year["order_date"]).dt.year
)

returns_year = (
    returns_year
    .groupby("year")
    .agg(
        Refunds=("valid_refund_amount", "sum"),
        ReturnShipping=("valid_return_shipping_cost", "sum"),
        Restocking=("valid_restocking_cost", "sum")
    )
    .reset_index()
)


# Shipping assigned to the year of the original order
shipping_year = (
    fulfillment[
        ["order_id", "shipping_cost"]
    ]
    .merge(
        order_profitability[
            ["order_id", "order_date"]
        ].drop_duplicates("order_id"),
        on="order_id",
        how="inner",
        validate="many_to_one"
    )
)

shipping_year["year"] = (
    pd.to_datetime(shipping_year["order_date"]).dt.year
)

shipping_year = (
    shipping_year
    .groupby("year")
    .agg(
        OutboundShipping=("shipping_cost", "sum")
    )
    .reset_index()
)


# Combine
contribution_year = (
    sales_year
    .merge(returns_year, on="year", how="left")
    .merge(shipping_year, on="year", how="left")
    .fillna(0)
)

contribution_year["ContributionProfit"] = (
    contribution_year["ProductMargin"]
    - contribution_year["Refunds"]
    - contribution_year["ReturnShipping"]
    - contribution_year["Restocking"]
    - contribution_year["OutboundShipping"]
)

contribution_year["ContributionMarginPct"] = (
    contribution_year["ContributionProfit"]
    / contribution_year["Revenue"]
    * 100
)

contribution_year["ContributionProfitGrowthPct"] = (
    contribution_year["ContributionProfit"]
    .pct_change()
    * 100
)

contribution_year

,year,Revenue,ProductCost,ProductMargin,Refunds,ReturnShipping,Restocking,OutboundShipping,ContributionProfit,ContributionMarginPct,ContributionProfitGrowthPct
0,2024,"4,035,042.19","1,984,582.97","2,050,459.22","317,579.04","57,149.10","12,672.82","143,005.13","1,520,053.13",37.67,NaN
1,2025,"3,963,540.95","1,946,646.72","2,016,894.23","323,901.64","58,014.64","13,001.91","140,471.73","1,481,504.31",37.38,-2.54


### Conclusion:
Contribution Profit declined by :`-2.54`. 

Product margin percentage barely improved:

`50.82% → 50.89%`

But contribution margin deteriorated:

`37.67% → 37.38%`

# Category Economics

### Research Question:
    Which categories generate the most revenue, and which retain the most value?


In [17]:
category=(
    order_profitability
    .groupby('category')
    .agg(
        Revenue=('gross_revenue','sum'),
        ProductCost=('product_cost','sum'),
        Units=('quantity','sum'),
        Orders=('order_id','nunique')
    ).reset_index()
)

category['ProductMargin']=( category['Revenue'] - category['ProductCost'] )

category['MarginPct']= ( category['ProductMargin'] / category['Revenue'] * 100 )

category.sort_values('Revenue', ascending=False)

,category,Revenue,ProductCost,Units,Orders,ProductMargin,MarginPct
1,Apparel,"1,683,707.90","951,663.62",39041,20338,"732,044.28",43.48
5,Home & Kitchen,"1,482,017.26","610,884.57",32202,17281,"871,132.69",58.78
3,Electronics,"1,346,858.87","439,246.34",30230,16258,"907,612.53",67.39
0,Accessories,"1,164,637.30","655,755.53",23163,12611,"508,881.77",43.69
2,Beauty,"921,443.22","538,926.30",21694,11911,"382,516.92",41.51
4,Footwear,"751,762.66","401,530.40",17322,9567,"350,232.26",46.59
6,Sports,"648,155.93","333,222.93",15840,8739,"314,933.00",48.59


### Add return economics to category

In [18]:
# ===========================================================
# CATEGORY ECONOMICS 
# ============================================================

# 1. Build the sales-side category table from scratch
category_sales = (
    order_profitability
    .groupby("category")
    .agg(
        Revenue=("gross_revenue", "sum"),
        ProductCost=("product_cost", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique")
    )
    .reset_index()
)

# Product margin
category_sales["ProductMargin"] = (
    category_sales["Revenue"]
    - category_sales["ProductCost"]
)

# Product margin %
category_sales["MarginPct"] = (
    category_sales["ProductMargin"]
    / category_sales["Revenue"]
    * 100
)

# ------------------------------------------------------------
# 2. Create ONE unique mapping from order_item -> category
# ------------------------------------------------------------

item_category = (
    order_profitability[
        ["order_item_id", "category"]
    ]
    .drop_duplicates("order_item_id")
)

# Safety check
assert item_category["order_item_id"].duplicated().sum() == 0


# ------------------------------------------------------------
# 3. Build return-side category table 
# ------------------------------------------------------------

return_category = (
    returns
    .merge(
        item_category,
        on="order_item_id",
        how="left",
        validate="many_to_one"
    )
    .groupby("category")
    .agg(
        ReturnedUnits=("valid_return_quantity", "sum"),
        Refunds=("valid_refund_amount", "sum"),
        ReturnShipping=("valid_return_shipping_cost", "sum"),
        Restocking=("valid_restocking_cost", "sum")
    )
    .reset_index()
)

# Total return-related cost
return_category["TotalReturnCost"] = (
    return_category["Refunds"]
    + return_category["ReturnShipping"]
    + return_category["Restocking"]
)


# ------------------------------------------------------------
# 4. Merge 
# ------------------------------------------------------------

category = category_sales.merge(
    return_category,
    on="category",
    how="left",
    validate="one_to_one"
)

# Categories with no returns get zero
return_columns = [
    "ReturnedUnits",
    "Refunds",
    "ReturnShipping",
    "Restocking",
    "TotalReturnCost"
]

category[return_columns] = category[return_columns].fillna(0)


# ------------------------------------------------------------
# 5. Return rate
# ------------------------------------------------------------

category["ReturnRatePct"] = (
    category["ReturnedUnits"]
    / category["Units"]
    * 100
)


# ------------------------------------------------------------
# 6. Sort
# ------------------------------------------------------------

category = category.sort_values(
    "ReturnRatePct",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Final column order
# ------------------------------------------------------------

category = category[
    [
        "category",
        "Revenue",
        "ProductCost",
        "ProductMargin",
        "MarginPct",
        "Units",
        "Orders",
        "ReturnedUnits",
        "ReturnRatePct",
        "Refunds",
        "ReturnShipping",
        "Restocking",
        "TotalReturnCost"
    ]
]


category

,category,Revenue,ProductCost,ProductMargin,MarginPct,Units,Orders,ReturnedUnits,ReturnRatePct,Refunds,ReturnShipping,Restocking,TotalReturnCost
0,Apparel,"1,683,707.90","951,663.62","732,044.28",43.48,39041,20338,4401,11.27,"190,046.91","34,849.70","7,623.16","232,519.77"
1,Footwear,"751,762.66","401,530.40","350,232.26",46.59,17322,9567,1791,10.34,"78,111.66","14,351.92","3,153.61","95,617.19"
2,Accessories,"1,164,637.30","655,755.53","508,881.77",43.69,23163,12611,1739,7.51,"84,612.44","13,806.83","3,356.22","101,775.49"
3,Electronics,"1,346,858.87","439,246.34","907,612.53",67.39,30230,16258,2088,6.91,"92,731.07","16,706.05","3,722.76","113,159.88"
4,Home & Kitchen,"1,482,017.26","610,884.57","871,132.69",58.78,32202,17281,2154,6.69,"99,905.64","17,036.60","4,010.81","120,953.05"
5,Sports,"648,155.93","333,222.93","314,933.00",48.59,15840,8739,1053,6.65,"41,816.50","8,323.23","1,651.68","51,791.41"
6,Beauty,"921,443.22","538,926.30","382,516.92",41.51,21694,11911,1284,5.92,"54,256.46","10,089.41","2,156.49","66,502.36"


## Conclusion:

Apparel is the clearest profitability pressure point. It generates the highest revenue but also has the highest return rate and the highest return-related cost. Electronics and Home & Kitchen retain much more value from each dollar of revenue.

## Find Profit Leakage Candidates

In [19]:
# This tells us: "Which categories have both meaningful revenue scale and relatively high return rates?"
revenue_threshold = category["Revenue"].quantile(0.75)
return_threshold = category["ReturnRatePct"].quantile(0.75)

high_scale_high_return = category[
    (category["Revenue"] >= revenue_threshold)
    &
    (category["ReturnRatePct"] >= return_threshold)
]

high_scale_high_return

,category,Revenue,ProductCost,ProductMargin,MarginPct,Units,Orders,ReturnedUnits,ReturnRatePct,Refunds,ReturnShipping,Restocking,TotalReturnCost
0,Apparel,"1,683,707.90","951,663.62","732,044.28",43.48,39041,20338,4401,11.27,"190,046.91","34,849.70","7,623.16","232,519.77"


## Conclusion:

Apparel is the clearest high-scale/high-return category and therefore deserves priority investigation.

## Product Level Investigation

In [20]:
product = (
    order_profitability
    .groupby(
        [
            "product_id",
            "product_name",
            "category"
        ]
    )
    .agg(
        Revenue=("gross_revenue", "sum"),
        ProductCost=("product_cost", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique")
    )
    .reset_index()
)

product["ProductMargin"] = (
    product["Revenue"]
    - product["ProductCost"]
)

product["MarginPct"] = (
    product["ProductMargin"]
    / product["Revenue"]
    * 100
)

In [21]:
# Add returns directly from the returns dataframe
product_return = (
    returns
    .groupby("product_id")
    .agg(
        ReturnedUnits=("valid_return_quantity", "sum"),
        ReturnCost=("valid_refund_amount", "sum")
    )
    .reset_index()
)

In [22]:
product_return = (
    returns.drop(columns=["product_id"], errors="ignore")
    .merge(
        order_profitability[
            [
                "order_item_id",
                "product_id"
            ]
        ],
        on="order_item_id",
        how="left"
    )
    .groupby("product_id")
    .agg(
        ReturnedUnits=("valid_return_quantity", "sum"),
        Refunds=("valid_refund_amount", "sum"),
        ReturnShipping=("valid_return_shipping_cost", "sum"),
        Restocking=("valid_restocking_cost", "sum")
    )
    .reset_index()
)

product_return["TotalReturnCost"] = (
    product_return["Refunds"]
    + product_return["ReturnShipping"]
    + product_return["Restocking"]
)

In [23]:
product = product.merge(
    product_return,
    on="product_id",
    how="left"
)

product["ReturnedUnits"] = (
    product["ReturnedUnits"].fillna(0)
)

product["TotalReturnCost"] = (
    product["TotalReturnCost"].fillna(0)
)

product["ReturnRatePct"] = (
    product["ReturnedUnits"]
    / product["Units"]
    * 100
)

In [24]:
#High volume/high return products
volume_threshold = product["Units"].quantile(0.75)

high_volume_products = product[
    product["Units"] >= volume_threshold
].sort_values(
    "ReturnRatePct",
    ascending=False
)

high_volume_products[
    [
        "product_id",
        "product_name",
        "category",
        "Units",
        "Revenue",
        "MarginPct",
        "ReturnedUnits",
        "ReturnRatePct",
        "TotalReturnCost"
    ]
].head(20)

,product_id,product_name,category,Units,Revenue,MarginPct,ReturnedUnits,ReturnRatePct,TotalReturnCost
265,P00266,Oak & Co. Sneakers 0266,Footwear,205,"15,062.03",46.16,34,16.59,"2,795.39"
572,P00573,Everline Dresses 0573,Apparel,231,"6,385.99",33.67,35,15.15,"1,272.21"
413,P00414,UrbanPeak Activewear 0414,Apparel,200,"16,270.24",49.84,30,15.00,"2,733.07"
734,P00735,Northstar Jeans 0735,Apparel,203,"4,278.47",48.64,29,14.29,817.58
444,P00445,Northstar Jeans 0445,Apparel,226,"5,487.44",34.36,32,14.16,"1,039.27"
512,P00513,Aster T-Shirts 0513,Apparel,236,"5,321.76",42.43,33,13.98,"1,026.02"
382,P00383,Everline Jeans 0383,Apparel,233,"25,206.78",34.98,31,13.30,"3,627.60"
370,P00371,Everline Dresses 0371,Apparel,201,"13,243.50",51.36,26,12.94,"1,927.72"
89,P00090,Vertex T-Shirts 0090,Apparel,201,"1,642.20",44.17,26,12.94,436.16
863,P00864,Oak & Co. Jackets 0864,Apparel,211,"6,199.53",54.54,27,12.80,"1,009.98"


In [25]:
product["ContributionBeforeOutboundShipping"] = (
    product["ProductMargin"]
    - product["TotalReturnCost"]
)

In [26]:
product[
    [
        "product_id",
        "category",
        "Units",
        "Revenue",
        "MarginPct",
        "ReturnedUnits",
        "ReturnRatePct",
        "TotalReturnCost",
        "ContributionBeforeOutboundShipping"
    ]
].sort_values(
    "TotalReturnCost",
    ascending=False
).head(20)

,product_id,category,Units,Revenue,MarginPct,ReturnedUnits,ReturnRatePct,TotalReturnCost,ContributionBeforeOutboundShipping
457,P00458,Footwear,225,"49,932.88",52.06,27,12.00,"6,412.51","19,581.09"
229,P00230,Apparel,181,"37,499.49",54.10,24,13.26,"5,318.32","14,968.85"
935,P00936,Footwear,207,"39,603.60",49.70,20,9.66,"4,049.13","15,635.40"
395,P00396,Apparel,227,"31,048.57",41.35,27,11.89,"4,035.38","8,802.73"
700,P00701,Apparel,181,"24,331.29",35.84,27,14.92,"3,939.91","4,779.21"
382,P00383,Apparel,233,"25,206.78",34.98,31,13.30,"3,627.60","5,190.55"
279,P00280,Electronics,200,"26,937.70",75.27,22,11.00,"3,271.20","17,004.02"
674,P00675,Home & Kitchen,189,"33,649.18",66.87,17,8.99,"3,250.10","19,250.38"
907,P00908,Accessories,200,"33,834.49",56.49,16,8.00,"2,917.41","16,195.90"
665,P00666,Apparel,209,"22,073.45",30.59,24,11.48,"2,851.68","3,900.04"


### Discount Economics

#### Research question
    Does deeper discounting coincide with weaker product economics?

In [27]:
discount = (
    order_profitability
    .groupby("discount_rate")
    .agg(
        Revenue=("gross_revenue", "sum"),
        ProductCost=("product_cost", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique")
    )
    .reset_index()
)

discount["ProductMargin"] = (
    discount["Revenue"]
    - discount["ProductCost"]
)

discount["MarginPct"] = (
    discount["ProductMargin"]
    / discount["Revenue"]
    * 100
)

In [28]:
discount

,discount_rate,Revenue,ProductCost,Units,Orders,ProductMargin,MarginPct
0,0.00,"4,075,121.04","1,898,874.39",86120,35029,"2,176,246.65",53.40
1,0.05,"1,622,146.31","787,868.18",35859,14402,"834,278.13",51.43
2,0.10,"1,081,002.74","554,527.17",25433,10359,"526,475.57",48.70
3,0.15,"652,398.51","355,011.81",16382,6518,"297,386.70",45.58
4,0.20,"375,886.53","216,998.37",10231,4264,"158,888.16",42.27
5,0.25,"192,028.01","117,949.77",5467,2200,"74,078.24",38.58


## Conclusion:
Deeper discounts are strongly associated with weaker product margin economics in this dataset.

### Discount and returns together

In [29]:
#Creating item-level return flags:

item_returns = (
    returns
    .groupby("order_item_id")
    .agg(
        ReturnedUnits=("valid_return_quantity", "sum")
    )
    .reset_index()
)

discount_returns = order_profitability[
    [
        "order_item_id",
        "discount_rate",
        "quantity"
    ]
].merge(
    item_returns,
    on="order_item_id",
    how="left"
)

discount_returns["ReturnedUnits"] = (
    discount_returns["ReturnedUnits"]
    .fillna(0)
)

#================================================================
discount_returns["HasReturn"] = (
    discount_returns["ReturnedUnits"] > 0
)

# =================================================================
# Aggregate
discount_return = (
    discount_returns
    .groupby("discount_rate")
    .agg(
        Items=("order_item_id", "nunique"),
        ReturnedItems=("HasReturn", "sum"),
        PurchasedUnits=("quantity", "sum"),
        ReturnedUnits=("ReturnedUnits", "sum")
    )
    .reset_index()
)

discount_return["ReturnedItemRatePct"] = (
    discount_return["ReturnedItems"]
    / discount_return["Items"]
    * 100
)

discount_return

,discount_rate,Items,ReturnedItems,PurchasedUnits,ReturnedUnits,ReturnedItemRatePct
0,0.00,48932,6879,86120,"6,932.00",14.06
1,0.05,20203,2926,35859,"2,950.00",14.48
2,0.10,14489,2092,25433,"2,105.00",14.44
3,0.15,9224,1261,16382,"1,273.00",13.67
4,0.20,5947,832,10231,836.00,13.99
5,0.25,3092,412,5467,414.00,13.32


## Conclusion:

* Return behavior is relatively stable across discount levels, while margin declines substantially as discounts deepen. The main observed discount problem is therefore margin erosion rather than a clear increase in returns.
  
* If you are worried that giving discounts makes people return more items, don't worry—the data shows they don't. Instead, you should worry about your profits. Giving deep discounts is severely cutting into the money you actually get to keep.


## Coupon Analysis

### Research Question
    Do coupon-driven orders exhibit different economics from non-coupon orders?

In [30]:
coupon_analysis = (
    order_profitability
    .assign(
        CouponUsed=order_profitability["coupon_code"].astype(str).str.strip().ne("")
    )
    .groupby("CouponUsed")
    .agg(
        Orders=("order_id", "nunique"),
        Revenue=("gross_revenue", "sum"),
        ProductCost=("product_cost", "sum"),
        Units=("quantity", "sum")
    )
    .reset_index()
)

coupon_analysis["ProductMargin"] = (
    coupon_analysis["Revenue"]
    - coupon_analysis["ProductCost"]
)

coupon_analysis["MarginPct"] = (
    coupon_analysis["ProductMargin"]
    / coupon_analysis["Revenue"]
    * 100
)

coupon_analysis


,CouponUsed,Orders,Revenue,ProductCost,Units,ProductMargin,MarginPct
0,False,36557,"4,236,223.78","1,981,444.94",89911,"2,254,778.84",53.23
1,True,36215,"3,762,359.36","1,949,784.75",89581,"1,812,574.61",48.18


## Conclusion:

Coupon-driven orders retain less product margin than non-coupon orders. The difference is meaningful, but this is an observed association rather than proof that coupons caused the lower margin.

## Return reason economics

### Research question
    Which return reasons create the greatest economic burden?


In [31]:
return_reason = (
    returns
    .groupby("return_reason")
    .agg(
        Events=("return_id", "count"),
        ReturnedUnits=("valid_return_quantity", "sum"),
        Refunds=("valid_refund_amount", "sum"),
        ReturnShipping=("valid_return_shipping_cost", "sum"),
        Restocking=("valid_restocking_cost", "sum")
    )
    .reset_index()
)

return_reason["TotalReturnCost"] = (
    return_reason["Refunds"]
    + return_reason["ReturnShipping"]
    + return_reason["Restocking"]
)

return_reason["CostPerReturnedUnit"] = (
    return_reason["TotalReturnCost"]
    / return_reason["ReturnedUnits"]
)

return_reason.sort_values(
    "TotalReturnCost",
    ascending=False
)

,return_reason,Events,ReturnedUnits,Refunds,ReturnShipping,Restocking,TotalReturnCost,CostPerReturnedUnit
0,Changed Mind,3050,3056,"135,105.88","24,446.89","5,390.88","164,943.65",53.97
3,Not as Expected,2813,2815,"125,583.25","22,185.39","4,951.21","152,719.85",54.25
2,Late Delivery,2140,2147,"94,664.85","16,944.14","3,807.45","115,416.44",53.76
6,Wrong Size,1986,1995,"84,386.70","15,855.56","3,406.75","103,649.01",51.95
1,Damaged in Transit,1619,1621,"73,249.44","12,842.38","2,928.86","89,020.68",54.92
4,Product Defect,1450,1455,"64,713.25","11,675.68","2,610.71","78,999.64",54.30
5,Wrong Item,1421,1421,"63,777.31","11,213.70","2,578.87","77,569.88",54.59


## Conclusion:

The biggest return burden is not one operational failure. It is concentrated in customer-expectation-related reasons, especially Changed Mind and Not as Expected. Together they account for about 41% of return-related cost.

## Fullfilment Analysis

### Research Analysis
    How much fulfillment friction exists in the dataset?

In [32]:
fulfillment["delivery_status"].value_counts()

delivery_status
On Time    18260
Late       11740
Name: count, dtype: int64

In [33]:
fulfillment_summary = (
    fulfillment
    .groupby("delivery_status")
    .agg(
        Shipments=("shipment_id", "count"),
        ShippingCost=("shipping_cost", "sum"),
        AvgShippingCost=("shipping_cost", "mean"),
        AvgDeliveryDays=("DeliveryDays", "mean")
    )
    .reset_index()
)

fulfillment_summary

,delivery_status,Shipments,ShippingCost,AvgShippingCost,AvgDeliveryDays
0,Late,11740,"114,153.94",9.72,3.88
1,On Time,18260,"169,322.92",9.27,2.14


## Late delivery rate

In [34]:
late_rate = (
    fulfillment["IsLate"].mean()
    * 100
)

print(f"Late delivery rate: {late_rate:.2f}%")

Late delivery rate: 39.13%


## Conclusion:
Nearly 4 in 10 tracked shipments were late, so fulfillment reliability is a significant operational issue.

## Late delivery vs returns

In [35]:
late_return = (
    fulfillment
    .groupby("delivery_status")
    .agg(
        Shipments=("shipment_id", "count"),
        ReturnedOrders=("HasReturn", "sum")
    )
    .reset_index()
)

late_return["ReturnRatePct"] = (
    late_return["ReturnedOrders"]
    / late_return["Shipments"]
    * 100
)

late_return

,delivery_status,Shipments,ReturnedOrders,ReturnRatePct
0,Late,11740,2203,18.76
1,On Time,18260,3443,18.86


## Conclusion:
* Late delivery is a major operational issue, but the shipment-level data does not show a meaningful difference in overall return incidence between late and on-time shipments.

`But we have any intreseting point also , that is:

* Some customers explicitly return because of late delivery, but late shipments overall are not associated with a higher total return rate.

In [36]:
fulfillment.groupby(
    ["shipping_method", "delivery_status"]
).agg(
    Shipments=("shipment_id", "count"),
    ShippingCost=("shipping_cost", "sum"),
    ReturnRate=("HasReturn", "mean")
).reset_index()

,shipping_method,delivery_status,Shipments,ShippingCost,ReturnRate
0,Express,Late,2597,"35,129.90",0.20
1,Express,On Time,3671,"49,637.06",0.18
2,Next-Day,Late,1312,"28,319.78",0.18
3,Next-Day,On Time,1851,"39,673.39",0.19
4,Pickup,Late,2172,"5,413.25",0.19
5,Pickup,On Time,3996,"10,048.92",0.19
6,Standard,Late,5659,"45,291.01",0.18
7,Standard,On Time,8742,"69,963.55",0.19


In [37]:
fulfillment_by_method = (
    fulfillment
    .groupby("shipping_method")
    .agg(
        Shipments=("shipment_id", "count"),
        ShippingCost=("shipping_cost", "sum"),
        AvgShippingCost=("shipping_cost", "mean"),
        LateRate=("IsLate", "mean"),
        ReturnRate=("HasReturn", "mean")
    )
    .reset_index()
)

fulfillment_by_method["LateRatePct"] = (
    fulfillment_by_method["LateRate"] * 100
)

fulfillment_by_method["ReturnRatePct"] = (
    fulfillment_by_method["ReturnRate"] * 100
)

fulfillment_by_method

,shipping_method,Shipments,ShippingCost,AvgShippingCost,LateRate,ReturnRate,LateRatePct,ReturnRatePct
0,Express,6268,"84,766.96",13.52,0.41,0.19,41.43,18.79
1,Next-Day,3163,"67,993.17",21.50,0.41,0.18,41.48,18.40
2,Pickup,6168,"15,462.17",2.51,0.35,0.19,35.21,18.95
3,Standard,14401,"115,254.56",8.00,0.39,0.19,39.30,18.87


## Warehouse investigation

### Research question:
    Are delivery outcomes concentrated in particular warehouses?

In [38]:
warehouse = (
    fulfillment
    .groupby("warehouse_id")
    .agg(
        Shipments=("shipment_id", "count"),
        ShippingCost=("shipping_cost", "sum"),
        AvgShippingCost=("shipping_cost", "mean"),
        LateRate=("IsLate", "mean"),
        ReturnRate=("HasReturn", "mean")
    )
    .reset_index()
)

warehouse["LateRatePct"] = warehouse["LateRate"] * 100
warehouse["ReturnRatePct"] = warehouse["ReturnRate"] * 100

warehouse.sort_values(
    "LateRatePct",
    ascending=False
)

,warehouse_id,Shipments,ShippingCost,AvgShippingCost,LateRate,ReturnRate,LateRatePct,ReturnRatePct
2,WH03,3963,"37,297.66",9.41,0.40,0.19,39.62,18.72
5,WH06,3625,"34,508.11",9.52,0.40,0.19,39.56,19.20
3,WH04,3153,"29,605.43",9.39,0.40,0.18,39.55,18.46
6,WH07,3960,"37,609.74",9.50,0.39,0.19,39.34,18.86
0,WH01,3635,"34,222.87",9.41,0.39,0.19,39.34,18.73
1,WH02,3879,"36,961.20",9.53,0.39,0.18,38.98,17.79
7,WH08,3670,"34,454.39",9.39,0.38,0.19,38.45,19.21
4,WH05,4115,"38,817.46",9.43,0.38,0.20,38.35,19.51


## Conclusion:
Fulfillment problems are not concentrated in one warehouse. Performance is fairly consistent across warehouses, so the issue appears broader than a single warehouse

## Customer Economics

### Research question:
    Do repeat customers generate stronger economics?

In [39]:
customer_segment = (
    customer_economics
    .groupby("CustomerSegment")
    .agg(
        Customers=("customer_id", "count"),
        Orders=("CompletedOrders", "sum"),
        Revenue=("Revenue", "sum"),
        ProductMargin=("ProductMargin", "sum"),
        Refunds=("Refunds", "sum"),
        ReturnShipping=("ReturnShippingCost", "sum"),
        Restocking=("RestockingCost", "sum"),
        Contribution=(
            "ContributionBeforeOutboundShipping",
            "sum"
        )
    )
    .reset_index()
)

customer_segment

,CustomerSegment,Customers,Orders,Revenue,ProductMargin,Refunds,ReturnShipping,Restocking,Contribution
0,Highly Repeat,4066,25533,"3,196,326.22","1,627,531.93","221,459.31","40,036.46","8,914.61","1,357,121.55"
1,No Completed Orders,4178,0,0.00,0.00,0.00,0.00,0.00,0.00
2,One-Time,7244,7244,"689,925.42","351,875.07","66,214.40","11,746.98","2,570.01","271,343.68"
3,Repeat,14512,39995,"4,112,331.50","2,087,946.45","353,806.97","63,380.30","14,190.11","1,656,569.07"


In [40]:
#Per customer metrice
customer_segment["RevenuePerCustomer"] = (
    customer_segment["Revenue"]
    / customer_segment["Customers"]
)

customer_segment["ContributionPerCustomer"] = (
    customer_segment["Contribution"]
    / customer_segment["Customers"]
)

customer_segment

,CustomerSegment,Customers,Orders,Revenue,ProductMargin,Refunds,ReturnShipping,Restocking,Contribution,RevenuePerCustomer,ContributionPerCustomer
0,Highly Repeat,4066,25533,"3,196,326.22","1,627,531.93","221,459.31","40,036.46","8,914.61","1,357,121.55",786.11,333.77
1,No Completed Orders,4178,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,One-Time,7244,7244,"689,925.42","351,875.07","66,214.40","11,746.98","2,570.01","271,343.68",95.24,37.46
3,Repeat,14512,39995,"4,112,331.50","2,087,946.45","353,806.97","63,380.30","14,190.11","1,656,569.07",283.37,114.15


## Conclusion:
Repeat purchasing is strongly associated with better customer economics. The business gets most of its contribution from customers who purchase more than once.

## Customer type ecomonics


In [41]:
customer_type = (
    customer_economics
    .groupby("customer_type")
    .agg(
        Customers=("customer_id", "count"),
        Orders=("CompletedOrders", "sum"),
        Revenue=("Revenue", "sum"),
        ProductMargin=("ProductMargin", "sum"),
        Contribution=(
            "ContributionBeforeOutboundShipping",
            "sum"
        ),
        Refunds=("Refunds", "sum")
    )
    .reset_index()
)

customer_type["RevenuePerCustomer"] = (
    customer_type["Revenue"]
    / customer_type["Customers"]
)

customer_type["ContributionPerCustomer"] = (
    customer_type["Contribution"]
    / customer_type["Customers"]
)

customer_type

,customer_type,Customers,Orders,Revenue,ProductMargin,Contribution,Refunds,RevenuePerCustomer,ContributionPerCustomer
0,Consumer,26296,58721,"5,030,459.47","2,556,812.50","1,927,277.32","516,324.10",191.30,73.29
1,Small Business,3704,14051,"2,968,123.67","1,510,540.95","1,357,756.98","125,156.58",801.33,366.57


## Conclusion: 
Small Business customers are a disproportionately valuable segment. They represent a small share of customers but contribute much more per customer than consumers.

## Loyalty Economics

In [42]:
loyalty = (
    customer_economics
    .groupby("loyalty_tier")
    .agg(
        Customers=("customer_id", "count"),
        Orders=("CompletedOrders", "sum"),
        Revenue=("Revenue", "sum"),
        ProductMargin=("ProductMargin", "sum"),
        Contribution=(
            "ContributionBeforeOutboundShipping",
            "sum"
        ),
        Refunds=("Refunds", "sum")
    )
    .reset_index()
)

loyalty["RevenuePerCustomer"] = (
    loyalty["Revenue"]
    / loyalty["Customers"]
)

loyalty["ContributionPerCustomer"] = (
    loyalty["Contribution"]
    / loyalty["Customers"]
)

loyalty

,loyalty_tier,Customers,Orders,Revenue,ProductMargin,Contribution,Refunds,RevenuePerCustomer,ContributionPerCustomer
0,Bronze,17976,20352,"1,969,666.88","1,002,889.70","781,199.81","182,007.08",109.57,43.46
1,Gold,2056,14008,"1,729,486.17","878,935.33","730,880.52","121,109.33",841.19,355.49
2,Platinum,234,2645,"466,513.39","235,786.60","206,714.92","23,901.22","1,993.65",883.40
3,Silver,9734,35767,"3,832,916.70","1,949,741.82","1,566,239.05","314,463.05",393.77,160.90


## Conclusion:
Higher loyalty tiers are strongly associated with higher customer value, with Platinum customers producing substantially more contribution per customer than Bronze customers.

## Acquisition channel economics

In [43]:
acquisition = (
    customer_economics
    .groupby("acquisition_channel")
    .agg(
        Customers=("customer_id", "count"),
        Orders=("CompletedOrders", "sum"),
        Revenue=("Revenue", "sum"),
        Contribution=(
            "ContributionBeforeOutboundShipping",
            "sum"
        )
    )
    .reset_index()
)

acquisition["RevenuePerCustomer"] = (
    acquisition["Revenue"]
    / acquisition["Customers"]
)

acquisition["ContributionPerCustomer"] = (
    acquisition["Contribution"]
    / acquisition["Customers"]
)

acquisition

,acquisition_channel,Customers,Orders,Revenue,Contribution,RevenuePerCustomer,ContributionPerCustomer
0,Affiliate,3011,6041,"663,677.36","268,162.73",220.42,89.06
1,Direct,4820,13259,"1,467,324.55","601,591.91",304.42,124.81
2,Email,4191,11656,"1,286,071.09","536,193.97",306.86,127.94
3,Google Ads,6662,14966,"1,635,243.87","668,176.91",245.46,100.30
4,Meta,5968,12960,"1,422,778.41","587,111.37",238.40,98.38
5,Organic,5348,13890,"1,523,487.86","623,797.41",284.87,116.64


## Conclusion
Email-acquired customers show the highest contribution per customer in this dataset, while Affiliate customers show the lowest. However, this is not CAC-adjusted, so it should not be used alone to decide marketing investment.

## Marketing Economics

In [44]:
marketing_channel = (
    marketing
    .groupby("channel")
    .agg(
        Spend=("ad_spend", "sum"),
        AttributedRevenue=("attributed_revenue", "sum"),
        NewCustomers=("new_customers", "sum"),
        AttributedOrders=("attributed_orders", "sum"),
        Impressions=("impressions", "sum"),
        Clicks=("clicks", "sum")
    )
    .reset_index()
)

marketing_channel["AttributedROAS"] = (
    marketing_channel["AttributedRevenue"]
    / marketing_channel["Spend"]
)

marketing_channel["CostPerNewCustomer"] = (
    marketing_channel["Spend"]
    / marketing_channel["NewCustomers"]
)

marketing_channel["CTR"] = (
    marketing_channel["Clicks"]
    / marketing_channel["Impressions"]
    * 100
)

marketing_channel

,channel,Spend,AttributedRevenue,NewCustomers,AttributedOrders,Impressions,Clicks,AttributedROAS,CostPerNewCustomer,CTR
0,Affiliate,"4,679,496.20","15,838,161.50",92290,173565,346781311,5291664,3.38,50.70,1.53
1,Display,"2,795,257.39","4,881,430.86",28644,53463,313420607,2924098,1.75,97.59,0.93
2,Email,"4,025,337.87","82,763,197.95",492089,917964,346845292,12394679,20.56,8.18,3.57
3,Google Ads,"10,271,276.46","36,917,634.32",215178,404637,389145564,8822698,3.59,47.73,2.27
4,Meta,"4,238,231.25","20,373,945.18",120701,226032,312877555,5786230,4.81,35.11,1.85


In [53]:
## Marketing Attribution reconciliation

# ============================================================
# MARKETING ATTRIBUTION RECONCILIATION
# ============================================================

actual_revenue = order_profitability["gross_revenue"].sum()

attributed_revenue = marketing["attributed_revenue"].sum()

reported_new_customers = marketing["new_customers"].sum()

customer_count = customer_economics["customer_id"].nunique()

marketing_check = pd.DataFrame({
    "Metric": [
        "Actual completed revenue",
        "Attributed marketing revenue",
        "Attributed revenue / actual revenue",
        "Reported marketing new customers",
        "Customer dimension count"
    ],
    "Value": [
        actual_revenue,
        attributed_revenue,
        attributed_revenue / actual_revenue,
        reported_new_customers,
        customer_count
    ]
})

marketing_check



,Metric,Value
0,Actual completed revenue,"7,998,583.14"
1,Attributed marketing revenue,"160,774,369.81"
2,Attributed revenue / actual revenue,20.10
3,Reported marketing new customers,"948,902.00"
4,Customer dimension count,"30,000.00"


## Conclusion:
Marketing revenue and new-customer metrics are attribution-level measures and do not reconcile directly to the transaction/customer tables. Therefore ROAS and cost-per-reported-new-customer are used as attributed efficiency indicators, not actual incremental revenue or unique CAC.

## Marketing campaigns type

### Research question
    Does campaign objective/type correspond to different attributed economics?

In [45]:
campaign_type = (
    marketing
    .groupby(
        [
            "campaign_type",
            "objective"
        ]
    )
    .agg(
        Spend=("ad_spend", "sum"),
        AttributedRevenue=("attributed_revenue", "sum"),
        NewCustomers=("new_customers", "sum"),
        AttributedOrders=("attributed_orders", "sum")
    )
    .reset_index()
)

campaign_type["AttributedROAS"] = (
    campaign_type["AttributedRevenue"]
    / campaign_type["Spend"]
)

campaign_type["CostPerNewCustomer"] = (
    campaign_type["Spend"]
    / campaign_type["NewCustomers"]
)

campaign_type

,campaign_type,objective,Spend,AttributedRevenue,NewCustomers,AttributedOrders,AttributedROAS,CostPerNewCustomer
0,Acquisition,New Customer,"1,135,477.15","2,470,705.92",14351,27118,2.18,79.12
1,Acquisition,Revenue,"7,367,097.74","45,169,027.46",262796,493089,6.13,28.03
2,Brand,Awareness,"796,763.42","3,815,832.05",22796,42822,4.79,34.95
3,Brand,New Customer,"3,402,446.24","21,892,782.36",130162,244446,6.43,26.14
4,Promotional,New Customer,"4,337,657.74","19,727,602.78",116123,216254,4.55,37.35
5,Promotional,Repeat Purchase,"1,725,167.27","4,972,755.08",29479,55489,2.88,58.52
6,Promotional,Revenue,"5,877,051.93","40,186,299.74",239489,448914,6.84,24.54
7,Retention,Repeat Purchase,"1,367,937.68","22,539,364.42",133706,247529,16.48,10.23


## Marketing caveat

Marketing revenue is attributed revenue.
It should not be interpreted as incremental revenue caused by the campaign.
ROAS therefore represents attributed ROAS rather than causal/incremental ROAS.

## Sales Channel Economics

### Research Question:
    Do any sales channels materially underperform after return-related costs?

In [55]:
# ============================================================
# SALES CHANNEL CONTRIBUTION ECONOMICS
# ============================================================

channel_sales = (
    order_profitability
    .groupby("sales_channel")
    .agg(
        Revenue=("gross_revenue", "sum"),
        ProductCost=("product_cost", "sum"),
        ProductMargin=("ProductMargin", "sum"),
        Orders=("order_id", "nunique"),
        Units=("quantity", "sum")
    )
    .reset_index()
)

channel_returns = (
    returns
    .merge(
        order_profitability[
            ["order_id", "sales_channel"]
        ].drop_duplicates("order_id"),
        on="order_id",
        how="left",
        validate="many_to_one"
    )
    .groupby("sales_channel")
    .agg(
        ReturnedUnits=("valid_return_quantity", "sum"),
        Refunds=("valid_refund_amount", "sum"),
        ReturnShipping=("valid_return_shipping_cost", "sum"),
        Restocking=("valid_restocking_cost", "sum")
    )
    .reset_index()
)

channel = channel_sales.merge(
    channel_returns,
    on="sales_channel",
    how="left"
).fillna(0)

channel["TotalReturnCost"] = (
    channel["Refunds"]
    + channel["ReturnShipping"]
    + channel["Restocking"]
)

channel["ContributionBeforeOutboundShipping"] = (
    channel["ProductMargin"]
    - channel["TotalReturnCost"]
)

channel["ContributionMarginPct"] = (
    channel["ContributionBeforeOutboundShipping"]
    / channel["Revenue"]
    * 100
)

channel["ReturnRatePct"] = (
    channel["ReturnedUnits"]
    / channel["Units"]
    * 100
)

channel.sort_values(
    "ContributionBeforeOutboundShipping",
    ascending=False
)

,sales_channel,Revenue,ProductCost,ProductMargin,Orders,Units,ReturnedUnits,Refunds,ReturnShipping,Restocking,TotalReturnCost,ContributionBeforeOutboundShipping,ContributionMarginPct,ReturnRatePct
3,Website,"3,424,930.17","1,682,681.59","1,742,248.58",31289,76789,6204,"276,673.18","49,383.49","11,041.78","337,098.45","1,405,150.13",41.03,8.08
2,Physical Store,"1,988,528.10","978,583.45","1,009,944.65",18145,44879,3625,"159,238.96","28,837.85","6,407.81","194,484.62","815,460.03",41.01,8.08
1,Mobile App,"1,799,328.40","884,405.06","914,923.34",16171,40226,3236,"141,924.39","25,492.81","5,725.01","173,142.21","741,781.13",41.23,8.04
0,Marketplace,"785,796.47","385,559.59","400,236.88",7167,17598,1445,"63,644.15","11,449.59","2,500.13","77,593.87","322,643.01",41.06,8.21


## Conclusion:
Sales-channel economics are surprisingly consistent. No channel appears to be a major profitability leak, so channel is not a priority compared with returns, discounting and customer economics.


## Opportunity sizing

Now we investigate: "What could potentially improve?"

In [46]:
#Start with return cost
return_reason

,return_reason,Events,ReturnedUnits,Refunds,ReturnShipping,Restocking,TotalReturnCost,CostPerReturnedUnit
0,Changed Mind,3050,3056,"135,105.88","24,446.89","5,390.88","164,943.65",53.97
1,Damaged in Transit,1619,1621,"73,249.44","12,842.38","2,928.86","89,020.68",54.92
2,Late Delivery,2140,2147,"94,664.85","16,944.14","3,807.45","115,416.44",53.76
3,Not as Expected,2813,2815,"125,583.25","22,185.39","4,951.21","152,719.85",54.25
4,Product Defect,1450,1455,"64,713.25","11,675.68","2,610.71","78,999.64",54.30
5,Wrong Item,1421,1421,"63,777.31","11,213.70","2,578.87","77,569.88",54.59
6,Wrong Size,1986,1995,"84,386.70","15,855.56","3,406.75","103,649.01",51.95


In [47]:
top_return_costs = return_reason.sort_values(
    "TotalReturnCost",
    ascending=False
)

top_return_costs

,return_reason,Events,ReturnedUnits,Refunds,ReturnShipping,Restocking,TotalReturnCost,CostPerReturnedUnit
0,Changed Mind,3050,3056,"135,105.88","24,446.89","5,390.88","164,943.65",53.97
3,Not as Expected,2813,2815,"125,583.25","22,185.39","4,951.21","152,719.85",54.25
2,Late Delivery,2140,2147,"94,664.85","16,944.14","3,807.45","115,416.44",53.76
6,Wrong Size,1986,1995,"84,386.70","15,855.56","3,406.75","103,649.01",51.95
1,Damaged in Transit,1619,1621,"73,249.44","12,842.38","2,928.86","89,020.68",54.92
4,Product Defect,1450,1455,"64,713.25","11,675.68","2,610.71","78,999.64",54.30
5,Wrong Item,1421,1421,"63,777.31","11,213.70","2,578.87","77,569.88",54.59


In [48]:
# Now we are creating scenrios to see , if we reduce our product Totalreturncost to 10, 15 and 20% then how much money can we save
opportunity = top_return_costs.copy()

for pct in [0.10, 0.15, 0.20]:
    opportunity[
        f"PotentialSavings_{int(pct*100)}Pct"
    ] = (
        opportunity["TotalReturnCost"]
        * pct
    )

opportunity

,return_reason,Events,ReturnedUnits,Refunds,ReturnShipping,Restocking,TotalReturnCost,CostPerReturnedUnit,PotentialSavings_10Pct,PotentialSavings_15Pct,PotentialSavings_20Pct
0,Changed Mind,3050,3056,"135,105.88","24,446.89","5,390.88","164,943.65",53.97,"16,494.37","24,741.55","32,988.73"
3,Not as Expected,2813,2815,"125,583.25","22,185.39","4,951.21","152,719.85",54.25,"15,271.99","22,907.98","30,543.97"
2,Late Delivery,2140,2147,"94,664.85","16,944.14","3,807.45","115,416.44",53.76,"11,541.64","17,312.47","23,083.29"
6,Wrong Size,1986,1995,"84,386.70","15,855.56","3,406.75","103,649.01",51.95,"10,364.90","15,547.35","20,729.80"
1,Damaged in Transit,1619,1621,"73,249.44","12,842.38","2,928.86","89,020.68",54.92,"8,902.07","13,353.10","17,804.14"
4,Product Defect,1450,1455,"64,713.25","11,675.68","2,610.71","78,999.64",54.30,"7,899.96","11,849.95","15,799.93"
5,Wrong Item,1421,1421,"63,777.31","11,213.70","2,578.87","77,569.88",54.59,"7,756.99","11,635.48","15,513.98"


## Modeled Potenial Savings 
(If we reduce product cost upto 10,15 or 20% then how much money can we save)

In [56]:
total_return_cost = (
    returns["valid_refund_amount"].sum()
    + returns["valid_return_shipping_cost"].sum()
    + returns["valid_restocking_cost"].sum()
)

overall_opportunity = pd.DataFrame({
    "ReductionScenario": ["10%", "15%", "20%"],
    "PotentialSavings": [
        total_return_cost * 0.10,
        total_return_cost * 0.15,
        total_return_cost * 0.20
    ]
})

overall_opportunity

,ReductionScenario,PotentialSavings
0,10%,"78,231.91"
1,15%,"117,347.87"
2,20%,"156,463.83"


## Conclusion:
A 10–20% reduction in return-related cost would represent a modeled opportunity of approximately $78K–$156K.

In [49]:
## Leakage Waterfall
waterfall = pd.DataFrame({
    "Metric": [
        "Revenue",
        "Product Cost",
        "Refunds",
        "Return Shipping",
        "Restocking",
        "Outbound Shipping",
        "Contribution Profit"
    ],
    "Amount": [
        baseline["Revenue"],
        -baseline["ProductCost"],
        -baseline["Refunds"],
        -baseline["ReturnShipping"],
        -baseline["Restocking"],
        -baseline["OutboundShipping"],
        contribution_profit
    ]
})

waterfall

,Metric,Amount
0,Revenue,"7,998,583.14"
1,Product Cost,"-3,931,229.69"
2,Refunds,"-641,480.68"
3,Return Shipping,"-115,163.74"
4,Restocking,"-25,674.73"
5,Outbound Shipping,"-283,476.86"
6,Contribution Profit,"3,001,557.44"


## Evidence Table

In [57]:
# Final evidence table
findings = pd.DataFrame([
    {
        "Area": "Profitability",
        "Question": "How much value remains after major direct costs?",
        "Finding": "The business generates $8.00M revenue and retains $3.00M contribution profit after product, return and outbound shipping costs.",
        "Business_Implication": "Direct operating leakage reduces revenue to a 37.53% contribution margin.",
        "Evidence": "$7,998,583 revenue | $3,001,557 contribution profit | 37.53% contribution margin"
    },
    {
        "Area": "Trend",
        "Question": "Did profitability improve or deteriorate over time?",
        "Finding": "Revenue declined 1.77% in 2025. Product margin percentage was almost unchanged, but contribution profit fell about 2.54%.",
        "Business_Implication": "The pressure is not visible in product margin alone; downstream costs matter.",
        "Evidence": "Revenue: -1.77% | Contribution profit: -2.54% | Contribution margin: 37.67% to 37.38%"
    },
    {
        "Area": "Category",
        "Question": "Which category is the clearest profitability pressure point?",
        "Finding": "Apparel has the highest revenue, highest return rate and highest return-related cost.",
        "Business_Implication": "Apparel deserves priority investigation.",
        "Evidence": "$1.684M revenue | 11.27% return rate | $232,519.77 return cost"
    },
    {
        "Area": "Returns",
        "Question": "Which return reasons create the greatest economic burden?",
        "Finding": "Changed Mind and Not as Expected are the two largest return-cost drivers.",
        "Business_Implication": "Customer-expectation-related returns represent a major opportunity area.",
        "Evidence": "$317,663.50 combined | 40.61% of total return cost"
    },
    {
        "Area": "Discounting",
        "Question": "Does deeper discounting coincide with weaker product economics?",
        "Finding": "Product margin percentage declines consistently as discount depth increases.",
        "Business_Implication": "Heavy discounting materially reduces observed margin economics.",
        "Evidence": "53.40% margin at 0% discount vs 38.58% at 25%"
    },
    {
        "Area": "Discounting",
        "Question": "Do deeper discounts also show higher return rates?",
        "Finding": "Return behavior remains relatively stable across discount bands.",
        "Business_Implication": "The clearer discount problem is margin erosion rather than increased returns.",
        "Evidence": "Item-level return rate remains roughly 13.3%–14.5%"
    },
    {
        "Area": "Coupons",
        "Question": "Do coupon orders exhibit different economics?",
        "Finding": "Coupon orders have lower product margin percentage than non-coupon orders.",
        "Business_Implication": "Coupon usage should be evaluated for incremental value rather than revenue alone.",
        "Evidence": "48.18% coupon margin vs 53.23% non-coupon margin"
    },
    {
        "Area": "Fulfillment",
        "Question": "How much fulfillment friction exists?",
        "Finding": "39.13% of tracked shipments were late.",
        "Business_Implication": "Fulfillment reliability is a meaningful operational issue.",
        "Evidence": "11,740 late shipments out of 30,000"
    },
    {
        "Area": "Fulfillment",
        "Question": "Do late shipments have higher overall return rates?",
        "Finding": "Late and on-time shipments have almost identical return rates.",
        "Business_Implication": "Late delivery is a service problem, but the data does not show it as a broad return-rate driver.",
        "Evidence": "18.76% late vs 18.86% on-time"
    },
    {
        "Area": "Customers",
        "Question": "Do repeat customers generate stronger economics?",
        "Finding": "Repeat customers contribute substantially more per customer than one-time customers.",
        "Business_Implication": "Retention and repeat purchasing are strongly associated with customer value.",
        "Evidence": "$37.46 one-time vs $114.15 repeat vs $333.77 highly repeat"
    },
    {
        "Area": "Customer Type",
        "Question": "Which customer type is economically stronger?",
        "Finding": "Small Business customers generate much higher contribution per customer than consumers.",
        "Business_Implication": "Small Business is a disproportionately valuable segment.",
        "Evidence": "$366.57 vs $73.29 contribution per customer"
    },
    {
        "Area": "Loyalty",
        "Question": "Do higher loyalty tiers generate stronger economics?",
        "Finding": "Contribution per customer rises sharply across loyalty tiers.",
        "Business_Implication": "Higher-value customer groups deserve retention attention.",
        "Evidence": "$43.46 Bronze vs $883.40 Platinum"
    },
    {
        "Area": "Marketing",
        "Question": "Which channel has the strongest attributed efficiency?",
        "Finding": "Email has the highest attributed ROAS in the marketing dataset.",
        "Business_Implication": "Email appears efficient on an attributed basis, but the result is not causal.",
        "Evidence": "Email attributed ROAS: 20.56x"
    },
    {
        "Area": "Marketing",
        "Question": "Can attributed marketing revenue be treated as actual revenue?",
        "Finding": "No. Attributed marketing revenue greatly exceeds actual completed revenue.",
        "Business_Implication": "Marketing ROAS must be treated as an attribution metric, not incremental company revenue.",
        "Evidence": "$160.77M attributed vs $8.00M actual revenue"
    },
    {
        "Area": "Opportunity",
        "Question": "What could potentially improve?",
        "Finding": "A 10%–20% reduction in return-related costs represents a modeled $78K–$156K opportunity.",
        "Business_Implication": "Even partial reduction in return leakage could materially improve contribution profit.",
        "Evidence": "$78,231.92 at 10% | $156,463.83 at 20%"
    }
])

findings

,Area,Question,Finding,Business_Implication,Evidence
0,Profitability,How much value remains after major direct costs?,The business generates $8.00M revenue and reta...,Direct operating leakage reduces revenue to a ...,"$7,998,583 revenue | $3,001,557 contribution p..."
1,Trend,Did profitability improve or deteriorate over ...,Revenue declined 1.77% in 2025. Product margin...,The pressure is not visible in product margin ...,Revenue: -1.77% | Contribution profit: -2.54% ...
2,Category,Which category is the clearest profitability p...,"Apparel has the highest revenue, highest retur...",Apparel deserves priority investigation.,"$1.684M revenue | 11.27% return rate | $232,51..."
3,Returns,Which return reasons create the greatest econo...,Changed Mind and Not as Expected are the two l...,Customer-expectation-related returns represent...,"$317,663.50 combined | 40.61% of total return ..."
4,Discounting,Does deeper discounting coincide with weaker p...,Product margin percentage declines consistentl...,Heavy discounting materially reduces observed ...,53.40% margin at 0% discount vs 38.58% at 25%
5,Discounting,Do deeper discounts also show higher return ra...,Return behavior remains relatively stable acro...,The clearer discount problem is margin erosion...,Item-level return rate remains roughly 13.3%–1...
6,Coupons,Do coupon orders exhibit different economics?,Coupon orders have lower product margin percen...,Coupon usage should be evaluated for increment...,48.18% coupon margin vs 53.23% non-coupon margin
7,Fulfillment,How much fulfillment friction exists?,39.13% of tracked shipments were late.,Fulfillment reliability is a meaningful operat...,"11,740 late shipments out of 30,000"
8,Fulfillment,Do late shipments have higher overall return r...,Late and on-time shipments have almost identic...,"Late delivery is a service problem, but the da...",18.76% late vs 18.86% on-time
9,Customers,Do repeat customers generate stronger economics?,Repeat customers contribute substantially more...,Retention and repeat purchasing are strongly a...,$37.46 one-time vs $114.15 repeat vs $333.77 h...
